# `with_structured_output()` — The LangChain Method for Structured Responses

## What does `with_structured_output()` do?

`with_structured_output(Schema)` wraps a model so that it always returns data in the shape you define, instead of raw text.

Under the hood, LangChain:
1. Serialises your schema (Pydantic or TypedDict) into a JSON schema
2. Passes it to the model as a format constraint
3. Parses the model's output back into your Python schema type

The result: instead of `AIMessage(content="The sentiment is negative...")`, you get `{"sentiment": "negative", "summary": "..."}` directly.

## Why is this so useful?

Structured output is the bridge between AI and code. Once you have the LLM's response as a typed object, you can:
- Store it in a database
- Drive application logic based on the values (`if review.sentiment == "negative": alert_team()`)
- Pass it to the next step in a pipeline
- Display it in a UI

Without structured output, you'd have to parse the LLM's text manually — fragile and unreliable.

## The two schema options

```python
# Option 1: Pydantic (validation + object access)
from pydantic import BaseModel
class Review(BaseModel):
    sentiment: str
    summary: str
model.with_structured_output(Review)  # returns Review object

# Option 2: TypedDict (simpler, dict access)
from typing import TypedDict
class Review(TypedDict):
    sentiment: str
    summary: str
model.with_structured_output(Review)  # returns plain dict
```

## What you'll learn in this notebook

- How `.with_structured_output()` works with a TypedDict schema
- How to access individual fields from the result
- How the bound model looks internally (the RunnableBinding + JsonOutputParser)
- How to combine this with a plain model call for comparison

## Prerequisites

- Ollama running with `qwen2.5:latest` pulled (or change the model name)
- Virtual environment activated

In [23]:
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from typing import TypedDict, Annotated,Literal

In [24]:
model = ChatOllama(model="qwen2.5:latest")

In [25]:
class Review(TypedDict):
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["positive", "negative", "neutral"], "The sentiment of the review, either positive, negative or neutral"]  

In [26]:
structured_response = model.with_structured_output(Review).invoke("This product sucksss!!!")
print(structured_response)

{'summary': "It seems like you're not satisfied with the product. Can you please provide more details so I can assist you better?", 'sentiment': 'negative'}


In [27]:
model.with_structured_output(Review)

RunnableBinding(bound=ChatOllama(model='qwen2.5:latest'), kwargs={'format': {'title': 'Review', 'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v in iterable:\n        d[k] = v\ndict(**kwargs) -> new dictionary initialized with the name=value pairs\n    in the keyword argument list.  For example:  dict(one=1, two=2)", 'type': 'object', 'properties': {'summary': {'default': 'A brief summary of the review', 'type': 'string'}, 'sentiment': {'default': 'The sentiment of the review, either positive, negative or neutral', 'enum': ['positive', 'negative', 'neutral'], 'type': 'string'}}, 'required': ['summary', 'sentiment']}, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema'}, 'schema': {'title': 'Review', 'description': "dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a map

In [28]:
type(structured_response)

dict

In [29]:
structured_response["summary"]

"It seems like you're not satisfied with the product. Can you please provide more details so I can assist you better?"

In [30]:
structured_response["sentiment"]

'negative'

In [31]:
model.invoke("What is the capital of France?")

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-03-10T10:24:14.621971Z', 'done': True, 'done_reason': 'stop', 'total_duration': 266060916, 'load_duration': 78669458, 'prompt_eval_count': 36, 'prompt_eval_duration': 105927750, 'eval_count': 8, 'eval_duration': 76578584, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, id='lc_run--019cd746-9a12-7f43-b33b-267234da33d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 8, 'total_tokens': 44})